# IMU Configuration Analysis — PyTorch CNN

**ME594 Gait Phase Estimation** · Offline experimental comparison · NVIDIA RTX 4000 Ada-ready

This notebook recreates the **format of the teammate’s analysis**: sensor configurations in rows; held-out participants and bilateral stance/swing accuracies in columns; participant mean and SD. It also includes macro-F1, stance/swing recall, and walking-speed summaries.

> Experimental validity: hold out **entire participants** (LOSO), never random overlapping windows. Missing walking speeds are not invented; missing sensors remove only the ineligible recordings for each configuration. For direct 6 vs 7 comparison, restrict both to the *identical eligible recordings*. Results from LOSO CV are not an untouched independent final test.


## 1 · Check environment and GPU

In Anaconda Prompt install PyTorch with a CUDA wheel selected at [PyTorch official installer](https://pytorch.org/get-started/locally/), then `pip install numpy pandas scikit-learn matplotlib jupyterlab ipykernel artifact-tool`.

The CNN automatically uses CUDA where available. The lab computer's NVIDIA RTX 4000 Ada is suitable; if CUDA isn't available the notebook can still run on CPU, but will be much slower.

In [ ]:
import sys, subprocess, os, json, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:',torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0))
    print('GPU VRAM (GB):',round(torch.cuda.get_device_properties(0).total_memory/1e9,1))

## 2 · Set paths and experiment options

Place labeled recordings inside `data/`, preserving filenames such as `20261112_1_5mph_Chethan_labeled.csv`. All required features are read from those CSVs. **No other CSV is needed to compute the workbook-style accuracy tables.**

In [ ]:
DATA_GLOB = 'data/*_labeled.csv'
RESULTS = 'results_main'
EPOCHS = 100
BATCH_SIZE = 32
SEEDS = [2001]         # Use e.g. [2001,2002,2003] to assess run variability.
CONFIGS = ['LeftShank','RightFoot','LeftShank+RightShank','6IMU','7IMU']
MATCHED_TRIALS = False  # True for direct paired 6 vs 7 comparisons.
REQUIRE_CUDA = False   # Set True on the Ada 4000 lab PC once CUDA is working.
TEST_SUBJECT = None    # e.g. 'SubjectF' — excludes subject from analysis, does NOT score them.

print('Files found:',len(list(Path('data').glob('*_labeled.csv'))))
print('Working folder:',Path.cwd())

## 3 · Inspect participant and sensor coverage first

The inventory is generated by the training script. This cell does not train a CNN.

In [ ]:
from imu_config_analysis import file_inventory
paths=sorted(Path('data').glob('*_labeled.csv'))
if not paths:
    print('Add labeled CSV files to the data/ folder before training.')
else:
    inventory=file_inventory(paths)
    display(inventory[['subject','speed_mph','available_sensors','missing_sensors','has_labels','file']])
    print('Subjects:',list(inventory.subject.unique()))

## 4 · Train and evaluate the selected sensor configurations

Each held-out subject receives an independently trained model initialized from scratch. The architecture and training settings match your 1–7 IMU PyTorch approach. The displayed epoch losses are **training losses**; do not use the held-out subject for model selection or early stopping.

**This cell performs real training.** Start with just `LeftShank` for a short trial if you'd like.

In [ ]:
cmd = [sys.executable, 'imu_config_analysis.py', '--data', DATA_GLOB,
       '--configs', *CONFIGS, '--epochs',str(EPOCHS),
       '--batch-size',str(BATCH_SIZE), '--output',RESULTS,
       '--seeds',*[str(s) for s in SEEDS]]
if REQUIRE_CUDA: cmd.append('--require-cuda')
if MATCHED_TRIALS: cmd.append('--matched-trials')
if TEST_SUBJECT: cmd += ['--test-subject',TEST_SUBJECT]
print('Running:', ' '.join(cmd))
# Uncomment the next line to start full training:
# subprocess.run(cmd, check=True)
print('Uncomment subprocess.run(...) above when you are ready to train.')

### Alternative: Run every row in the teammate's configuration workbook

This can take considerable GPU time. Leave `EPOCHS` fixed for fair comparison. Do not choose the best configuration on an independent final test subject.

In [ ]:
all_configs_cmd = [sys.executable,'imu_config_analysis.py','--data',DATA_GLOB,
                   '--workbook-configs','--epochs',str(EPOCHS),
                   '--batch-size',str(BATCH_SIZE),'--output','results_all']
print(' '.join(all_configs_cmd))
# subprocess.run(all_configs_cmd,check=True)

## 5 · Table matching your teammate's layout

The result table shows individual held-out participants for each leg, along with a participant-average accuracy and SD (percentage points). Empty cells indicate unavailable/ineligible comparisons.

In [ ]:
res=Path(RESULTS)
p=res/'accuracy_table_both_legs.csv'
if p.exists():
    wide=pd.read_csv(p)
    display(wide.style.format({c:'{:.2%}' for c in wide if 'leg' in c and 'SD' not in c},na_rep='—'))
else:
    print('Run section 4 first; expected',p)

## 6 · Accuracy comparison chart and participant variability

In [ ]:
p=res/'configuration_summary.csv'
if p.exists():
    summ=pd.read_csv(p)
    fig,ax=plt.subplots(figsize=(11,5))
    x=np.arange(len(summ));w=.38
    ax.bar(x-w/2,summ.L_accuracy_mean*100,w,label='Left leg')
    ax.bar(x+w/2,summ.R_accuracy_mean*100,w,label='Right leg')
    ax.set_xticks(x,summ.config,rotation=35,ha='right')
    ax.set_ylabel('Mean participant-held-out accuracy (%)')
    ax.set_ylim(0,100);ax.legend();ax.grid(axis='y',alpha=.15)
    plt.tight_layout();plt.show()
else:print('No result summary yet.')

## 7 · Per-participant accuracy and swing/stance recall

A model can have good accuracy while predicting **stance too often**. Review macro F1, swing recall, and stance recall independently for both legs.

In [ ]:
p=res/'held_out_participant_metrics.csv'
if p.exists():
    folds=pd.read_csv(p)
    display(folds[['config','held_out_subject','seed','n_test_windows',
                  'L_accuracy','R_accuracy','L_macro_f1','R_macro_f1',
                  'L_swing_recall','R_swing_recall']].round(3))
else:print('No held-out metrics yet.')

## 8 · Walking speed coverage and accuracy

Some participants lack one or more speeds. The script records the actual speed when it can parse the filename; missing speeds are **not filled in**.

In [ ]:
p=res/'metrics_by_speed.csv'
if p.exists():
    byspeed=pd.read_csv(p)
    display(byspeed[['config','held_out_subject','speed_mph','n_test_windows',
                     'L_accuracy','R_accuracy']].sort_values(['config','held_out_subject','speed_mph']).round(3))
else:print('No speed-level output yet.')

## 9 · Missing IMU / recording eligibility audit

Check this before comparing configurations. A 7-IMU model trained on fewer people is **not a controlled comparison** with a 6-IMU model trained on a larger cohort.

In [ ]:
p=res/'recording_eligibility.csv'
if p.exists():
    eligibility=pd.read_csv(p)
    display(eligibility.groupby(['config','status']).size().unstack(fill_value=0))
    display(eligibility[eligibility.status=='excluded'][['config','subject','file','reason']].head(30))
else:print('No eligibility audit yet.')

## 10 · Paired 1-IMU vs 6-IMU comparison

Match **the same held-out participants** and ideally **the same recordings/speeds**, then compare changes. This is descriptive with 3–5 subjects; use no significance claim from thousands of correlated windows.

In [ ]:
p=res/'held_out_participant_metrics.csv'
if p.exists():
    m=pd.read_csv(p)
    a=m[m.config=='LeftShank'].set_index(['held_out_subject','seed'])
    b=m[m.config=='6IMU'].set_index(['held_out_subject','seed'])
    paired=a[['L_accuracy','R_accuracy']].join(b[['L_accuracy','R_accuracy']],lsuffix='_one',rsuffix='_six',how='inner')
    if len(paired):
        paired['left_difference_pp']=100*(paired.L_accuracy_one-paired.L_accuracy_six)
        paired['right_difference_pp']=100*(paired.R_accuracy_one-paired.R_accuracy_six)
        display(paired.round(3))
        print('Mean paired changes (pp):',paired[['left_difference_pp','right_difference_pp']].mean().round(2).to_dict())
    else:print('Both LeftShank and 6IMU must be run.')
else:print('No metrics yet.')

## 11 · Export a formatted Excel workbook

Use this only after the analysis has produced CSV files.

In [ ]:
if (Path(RESULTS)/'accuracy_table_both_legs.csv').exists():
    subprocess.run([sys.executable,'export_accuracy_workbook.py','--results',RESULTS],check=True)
    print('Excel saved to',Path(RESULTS)/'imu_configuration_accuracy.xlsx')
else:print('Run the training analysis first.')

## Scientific interpretation and next steps

- **Evidence:** higher held-out-participant stance/swing accuracy indicates more robust classification on the recorded participants, not automatically better event timing or clinical performance.
- **Clinical dashboard:** model inference and offline gait metrics are separate; a later phase can calculate heel-strike/toe-off MAE using FSR event flags and dense stride-1 predictions.
- **Fair sensor comparison:** same participant and trial cohort, same model architecture, same preprocessing policy, paired predictions, multiple seeds if possible.
- **Final untouched testing:** after selecting sensors/hyperparameters, retrain on all development participants and evaluate an excluded participant once. This notebook does not pretend LOSO results are that final test.
